# 02 — Text Embeddings (E5-base) from `overview`

Encodes the synopsis column into L2-normalized embeddings.

In [ ]:
import os, json, joblib, numpy as np, pandas as pd
from pathlib import Path

ART_DIR   = Path(os.getenv("ART_DIR",   "~/boxoffice/artifacts")).expanduser()
DATA_DIR  = Path(os.getenv("DATA_DIR",  "~/boxoffice/data")).expanduser()
IMG_DIR   = Path(os.getenv("IMG_DIR",   "~/boxoffice/data/img")).expanduser()

TEXT_DIR  = Path(os.getenv("TEXT_DIR",  ART_DIR / "text_emb")).expanduser()
CLIP_DIR  = Path(os.getenv("CLIP_DIR",  ART_DIR / "clip_emb")).expanduser()
ROI_DIR   = Path(os.getenv("ROI_DIR",   ART_DIR / "roi_feat")).expanduser()

def load_json(path): return json.loads(Path(path).read_text(encoding="utf-8"))

# Handy helper
def _vec(path, dim):
    try:
        v = np.load(path); v = np.asarray(v, np.float32).ravel()
        if   v.size == dim: return v
        elif v.size >  dim: return v[:dim]
        else:               return np.pad(v, (0, dim - v.size))
    except Exception:
        return np.zeros(dim, np.float32)

In [ ]:
text_index = pd.read_csv(ART_DIR / "text_index.csv") if (ART_DIR/"text_index.csv").exists() else pd.DataFrame()
text_meta  = load_json(ART_DIR / "text_model_meta.json") if (ART_DIR/"text_model_meta.json").exists() else {}
# Example: vec = np.load(TEXT_DIR / f"{tmdb_id}.npy")

In [ ]:
## FOR RELOAD ONLY
text_index = pd.read_csv(ART_DIR / "text_index.csv")
text_meta  = load_json(ART_DIR / "text_model_meta.json")
# load a vector for id=tmdb_id:
# vec = np.load(TEXT_DIR / f"{tmdb_id}.npy")  # shape=(dim,)

In [3]:
import os, math
from pathlib import Path
import pandas as pd, numpy as np
from tqdm.auto import tqdm

MASTER_CSV = os.getenv("MASTER_CSV", os.path.expanduser("~/boxoffice/data/movies_master_preprocessed.csv"))
ART_DIR    = os.getenv("ART_DIR",    os.path.expanduser("~/boxoffice/artifacts"))
TEXT_DIR   = os.getenv("TEXT_DIR",   os.path.join(ART_DIR, "text_emb"))
TEXT_COL   = os.getenv("TEXT_COL",   "overview")
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "32"))

Path(TEXT_DIR).mkdir(parents=True, exist_ok=True)

print("MASTER_CSV:", MASTER_CSV)
print("TEXT_DIR:", TEXT_DIR)
print("TEXT_COL:", TEXT_COL)

MASTER_CSV: C:\Users\Vaishob/boxoffice/data/movies_master_preprocessed.csv
TEXT_DIR: C:\Users\Vaishob/boxoffice/artifacts\text_emb
TEXT_COL: overview


In [4]:
df = pd.read_csv(MASTER_CSV)
assert "tmdb_id" in df.columns, "tmdb_id missing"
assert TEXT_COL in df.columns, f"{TEXT_COL} missing"

texts = df[[ "tmdb_id", TEXT_COL ]].copy()
texts[TEXT_COL] = texts[TEXT_COL].fillna("").astype(str).str.strip()
mask = texts[TEXT_COL].str.len() > 0
texts = texts[mask].copy()
texts["tmdb_id"] = pd.to_numeric(texts["tmdb_id"], errors="coerce").astype("Int64")
texts = texts.dropna(subset=["tmdb_id"])
print("to encode:", len(texts))
texts.head(3)

to encode: 4778


,tmdb_id,overview
0,8871,The Grinch decides to rob Whoville of Christma...
1,955,With computer genius Luther Stickell at his si...
2,98,"In the wake of their mother's death, two small..."


In [5]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "intfloat/e5-base"
model = SentenceTransformer(model_name, device=device)
model.max_seq_length = 512
print("device:", device)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

device: cpu


In [6]:
def l2_normalize(x, eps=1e-9):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return (x / np.maximum(n, eps)).astype(np.float32)

all_ids = texts["tmdb_id"].astype(int).tolist()
all_txt = texts[TEXT_COL].tolist()

vecs = []
bs = BATCH_SIZE
for i in tqdm(range(0, len(all_txt), bs)):
    batch = all_txt[i:i+bs]
    emb = model.encode(batch, batch_size=bs, show_progress_bar=False, normalize_embeddings=False, convert_to_numpy=True, device=device)
    vecs.append(emb)

X = np.vstack(vecs)
X = l2_normalize(X)

index_rows = []
for tmdb_id, v in zip(all_ids, X):
    np.save(Path(TEXT_DIR, f"{tmdb_id}.npy"), v)
    index_rows.append({"tmdb_id": tmdb_id, "dim": int(v.shape[0])})

pd.DataFrame(index_rows).to_csv(Path(TEXT_DIR, "text_index.csv"), index=False)
print("saved:", len(index_rows), "embeddings @", TEXT_DIR)

  0%|          | 0/150 [00:00<?, ?it/s]

saved: 4778 embeddings @ C:\Users\Vaishob/boxoffice/artifacts\text_emb


In [7]:
cover = set(texts["tmdb_id"].astype(int).tolist())
master = pd.read_csv(MASTER_CSV)["tmdb_id"].dropna().astype("Int64").astype(int)
have = sum(1 for t in master if t in cover)
print(f"text coverage: {have}/{len(master)} = {have/len(master):.1%}")

text coverage: 4791/4823 = 99.3%


In [9]:
# Save an index of what I wrote, plus model meta
from glob import glob
text_ids = sorted(int(Path(p).stem) for p in glob(str(TEXT_DIR / "*.npy")))
pd.DataFrame({"tmdb_id": text_ids, "dim":[embeds.shape[1] if 'embeds' in globals() else 768]*len(text_ids)}).to_csv(
    ART_DIR / "text_index.csv", index=False
)
save_json({"model":"intfloat/e5-base","max_seq_len":512}, ART_DIR / "text_model_meta.json")